# ⚡ Część 2: WSGI vs ASGI - Sync vs Async

**Cel:** Zrozumieć dokładnie CZYM różni się WSGI od ASGI, CO jest asynchroniczne i KIEDY to ma znaczenie.

**Analogia przewodnia:** 🍽️ Synchroniczny vs Asynchroniczny kucharz

---

## 1. Problem WSGI - blokowanie się

### Jak działa WSGI (synchronicznie):

```python
# WSGI application (Django/Flask)
def get_user(user_id):
    # Zapytanie do bazy danych - CZEKA 100ms
    user = db.query(User).filter(id=user_id).first()  # ⏳ BLOKUJE się
    return user
```

### Co się dzieje:

```
Request 1: GET /users/1
    ↓
WSGI worker 1: Wywołuje get_user(1)
    ↓
Zapytanie do DB... ⏳ CZEKA 100ms (I/O operation)
    ↓
W TYM CZASIE worker 1 jest ZABLOKOWANY!
Nie może obsłużyć Request 2!
    ↓
Request 2: GET /users/2 → MUSI CZEKAĆ aż worker 1 się zwolni
                         LUB
                         Trafia do worker 2 (jeśli dostępny)
```

**Jeden worker = jeden request naraz.**

Jeśli request czeka na:
- Bazę danych (100ms)
- External API call (500ms)
- Odczyt z pliku (50ms)

**Worker STOI I CZEKA** (blokuje się) - nie robi nic innego!

---

### 🍽️ Analogia: Synchroniczny kucharz

**WSGI (sync kucharz):**

```
1. Kucharz zaczyna robić carbonara
   ↓
2. Wstawia makaron do wody (10 minut gotowania)
   ↓
3. 😴 STOI I PATRZY przez 10 minut jak makaron się gotuje
   ↓
   (Nie może w tym czasie zrobić nic innego!)
   ↓
4. Timer dzwoni - makaron gotowy
   ↓
5. Kończy danie
   ↓
6. DOPIERO TERAZ może zacząć kolejne zamówienie
```

**Problem:** Kucharz **marnuje czas** stojąc i czekając.

**Rozwiązanie WSGI:** Zatrudnij **wielu kucharzy** (workers)!
- Worker 1: robi carbonara (czeka na makaron)
- Worker 2: robi sałatkę (czeka na kurczaka)
- Worker 3: robi pizzę (czeka na piekarnik)

**Ale:**
❌ Każdy worker = osobny proces (pamięć, CPU)
❌ Drogie (10 workers = 10× więcej RAM)
❌ Ograniczona skalowalność

---

## 2. ASGI - asynchroniczne rozwiązanie

### Czym jest ASGI?

**ASGI** = **Asynchronous Server Gateway Interface**

To **następca WSGI** dla kodu **asynchronicznego** (async/await).

### Jak działa ASGI (asynchronicznie):

```python
# ASGI application (FastAPI/Starlette)
async def get_user(user_id):
    # Zapytanie do bazy - używa await
    user = await db.query(User).filter(id=user_id).first()  # ⏳ await - NIE blokuje!
    return user
```

### Co się dzieje:

```
Request 1: GET /users/1
    ↓
ASGI worker: Wywołuje get_user(1)
    ↓
await db.query(...) → wysyła zapytanie do DB
    ↓
⚡ AWAIT! Worker NIE czeka, tylko:
    ↓
Request 2: GET /users/2 → Ten SAM worker obsługuje Request 2!
    ↓
await db.query(...) → wysyła zapytanie do DB
    ↓
Request 3: GET /users/3 → Ten SAM worker obsługuje Request 3!
    ↓
...
    ↓
DB odpowiada dla Request 1 → worker wraca i kończy Request 1
DB odpowiada dla Request 2 → worker wraca i kończy Request 2
DB odpowiada dla Request 3 → worker wraca i kończy Request 3
```

**Jeden worker = WIELE requestów jednocześnie!**

---

### 🍽️ Analogia: Asynchroniczny kucharz

**ASGI (async kucharz):**

```
1. Kucharz zaczyna robić carbonara
   ↓
2. Wstawia makaron do wody (10 minut gotowania)
   ↓
3. ⚡ AWAIT! Mówi: "Wrócę jak będzie gotowe"
   ↓
4. W MIĘDZYCZASIE robi sałatkę (inne zamówienie)
   ↓
5. Kroi chleb (jeszcze inne zamówienie)
   ↓
6. Przygotowuje deser (kolejne zamówienie)
   ↓
7. Timer dzwoni - makaron gotowy!
   ↓
8. Wraca do carbonary i kończy danie
   ↓
9. Wraca do sałatki, chleba, deseru...
```

**Jeden kucharz obsługuje WIELE zamówień równolegle!**

✅ Nie marnuje czasu czekając
✅ Wydajniejszy
✅ Mniej workers potrzebnych

---

## 3. CO DOKŁADNIE jest asynchroniczne w ASGI?

### Kluczowe pytanie: Co może być `await`?

**Tylko operacje I/O (Input/Output):**

✅ **Operacje I/O** (mogą być async):
- Zapytania do bazy danych
- HTTP calls do external API
- Odczyt/zapis plików
- WebSocket komunikacja
- Redis/cache operations
- Email sending

❌ **Obliczenia CPU** (NIE mogą być async):
- Pętle (for, while)
- Obliczenia matematyczne
- Data processing
- Image manipulation

### Dlaczego tylko I/O?

**I/O operations = czekanie na coś zewnętrznego:**
- Czekasz na bazę danych → możesz robić coś innego
- Czekasz na HTTP response → możesz robić coś innego
- Czekasz na plik z dysku → możesz robić coś innego

**CPU operations = aktywna praca:**
- Przeliczasz 1000000 liczb → **musisz** je przeliczyć, nie możesz "odłożyć"
- Analizujesz obraz → **musisz** go przeanalizować

---

### Przykłady async vs sync:


In [ ]:
# ============================================================================
# PRZYKŁAD 1: Database query
# ============================================================================

# WSGI (sync) - BLOKUJE
def get_users_sync():
    users = db.query(User).all()  # Czeka 100ms - BLOKUJE worker
    return users

# ASGI (async) - NIE BLOKUJE
async def get_users_async():
    users = await db.query(User).all()  # Czeka 100ms - NIE blokuje, robi inne rzeczy
    return users

In [ ]:
# ============================================================================
# PRZYKŁAD 2: External API call
# ============================================================================

import requests
import httpx  # async HTTP library

# WSGI (sync) - BLOKUJE
def get_weather_sync(city):
    response = requests.get(f"https://api.weather.com/{city}")  # Czeka 500ms - BLOKUJE
    return response.json()

# ASGI (async) - NIE BLOKUJE
async def get_weather_async(city):
    async with httpx.AsyncClient() as client:
        response = await client.get(f"https://api.weather.com/{city}")  # Czeka 500ms - NIE blokuje
    return response.json()

In [ ]:
# ============================================================================
# PRZYKŁAD 3: Wiele I/O operations RÓWNOLEGLE
# ============================================================================

import asyncio

# WSGI (sync) - KOLEJNO (wolne!)
def get_dashboard_data_sync():
    users = db.query(User).all()          # 100ms
    orders = db.query(Order).all()        # 150ms
    products = db.query(Product).all()    # 120ms
    # ŁĄCZNIE: 100 + 150 + 120 = 370ms
    
    return {"users": users, "orders": orders, "products": products}

# ASGI (async) - RÓWNOLEGLE (szybkie!)
async def get_dashboard_data_async():
    # Wszystkie 3 zapytania JEDNOCZEŚNIE!
    users, orders, products = await asyncio.gather(
        db.query(User).all(),       # 100ms  ⎤
        db.query(Order).all(),      # 150ms  ⎬ RÓWNOCZEŚNIE!
        db.query(Product).all()     # 120ms  ⎦
    )
    # ŁĄCZNIE: max(100, 150, 120) = 150ms (nie 370ms!)
    
    return {"users": users, "orders": orders, "products": products}

### 🎯 KLUCZOWY WNIOSEK:

**ASGI pozwala na równoległe I/O operations w JEDNYM procesie!**

WSGI:
```
Request A → DB query (100ms) → CZEKA → Response A
                                 ↓
                         Worker ZABLOKOWANY
```

ASGI:
```
Request A → await DB query (100ms) → ...
  ↓
Request B → await DB query (100ms) → ...  ⎤
  ↓                                       ⎬ RÓWNOCZEŚNIE!
Request C → await DB query (100ms) → ...  ⎦
  ↓
DB odpowiada → kończymy A, B, C
```

---

## 4. Co DOKŁADNIE można zrobić z ASGI, czego NIE można z WSGI?

### ✅ ASGI potrafi (WSGI NIE):

#### 1. **WebSockets** - dwukierunkowa komunikacja w czasie rzeczywistym

```python
# TYLKO ASGI!
from fastapi import WebSocket

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    while True:
        # Czekaj na wiadomość od klienta
        data = await websocket.receive_text()
        # Wyślij odpowiedź
        await websocket.send_text(f"Echo: {data}")
```

**Use case:** Czat, live notifications, real-time dashboards

**WSGI:** ❌ Niemożliwe (nie obsługuje długich połączeń)

---

#### 2. **Server-Sent Events (SSE)** - streaming od serwera do klienta

```python
# TYLKO ASGI!
from fastapi.responses import StreamingResponse

@app.get("/stream")
async def stream_data():
    async def event_generator():
        for i in range(100):
            yield f"data: Event {i}\n\n"
            await asyncio.sleep(1)  # Wysyłaj co sekundę
    
    return StreamingResponse(event_generator(), media_type="text/event-stream")
```

**Use case:** Progress updates, live logs, stock prices

**WSGI:** ⚠️ Możliwe, ale blokuje worker (nieefektywne)

---

#### 3. **Long polling** - klient czeka długo na odpowiedź

```python
# ASGI - nie blokuje worker
@app.get("/wait-for-update")
async def long_poll():
    # Czekaj max 30 sekund na update
    for _ in range(30):
        update = await check_for_update()  # NIE blokuje
        if update:
            return update
        await asyncio.sleep(1)  # NIE blokuje worker
    return {"status": "no update"}
```

**WSGI:** ❌ Blokuje worker na 30 sekund (katastrofa!)

---

#### 4. **Równoległe I/O operations** - wiele zapytań jednocześnie

```python
# ASGI - 3 zapytania równolegle
@app.get("/dashboard")
async def dashboard():
    # Wszystkie 3 JEDNOCZEŚNIE!
    users, orders, stats = await asyncio.gather(
        fetch_users(),
        fetch_orders(),
        fetch_stats()
    )
    return {"users": users, "orders": orders, "stats": stats}
```

**WSGI:** ❌ Musi wykonać kolejno (3× wolniejsze)

---

#### 5. **HTTP/2 Server Push**

ASGI obsługuje natywnie HTTP/2 features.

**WSGI:** ❌ Tylko HTTP/1.1

---

## 5. Porównanie WSGI vs ASGI - tabela

| Aspekt | WSGI | ASGI |
|--------|------|------|
| **Typ** | Synchroniczny | Asynchroniczny |
| **Funkcja signature** | `def app(environ, start_response)` | `async def app(scope, receive, send)` |
| **Protokoły** | HTTP tylko | HTTP + WebSocket + HTTP/2 |
| **Concurrent requests** | 1 worker = 1 request | 1 worker = wiele requestów |
| **I/O operations** | Blokuje worker | `await` - nie blokuje |
| **WebSockets** | ❌ Nie | ✅ Tak |
| **Streaming** | ⚠️ Ograniczone | ✅ Pełne wsparcie |
| **Long polling** | ❌ Blokuje | ✅ Nie blokuje |
| **Równoległe I/O** | ❌ Kolejno | ✅ Równolegle (`asyncio.gather`) |
| **Przykłady serwerów** | Gunicorn, uWSGI, mod_wsgi | Uvicorn, Daphne, Hypercorn |
| **Frameworki** | Django, Flask (bez async) | FastAPI, Starlette, Django (3.0+) |
| **Wydajność I/O** | Niska (blokuje) | Wysoka (nie blokuje) |
| **Workers needed** | Więcej (każdy = 1 request) | Mniej (każdy = wiele requestów) |
| **Kiedy używać** | Tradycyjne web apps, CRUD API | Real-time, WebSockets, I/O-heavy apps |

---

## 6. Kiedy używać WSGI, a kiedy ASGI?

### ✅ Używaj WSGI gdy:

1. **Proste CRUD API**
   - Zwykłe GET/POST/PUT/DELETE
   - Bez real-time features
   - Django/Flask z tradycyjnym SQLAlchemy

2. **Legacy code**
   - Istniejąca aplikacja Django/Flask
   - Biblioteki tylko sync

3. **CPU-intensive tasks**
   - Obliczenia, data processing
   - Async nie pomoże (to nie I/O!)

4. **Zespół nie zna async**
   - Async/await to nowa składnia
   - Wymaga zmiany myślenia

---

### ✅ Używaj ASGI gdy:

1. **WebSockets / Real-time**
   - Czat, notifications
   - Live dashboards
   - Multiplayer games

2. **I/O-heavy workloads**
   - Wiele DB queries
   - External API calls
   - File operations

3. **Streaming / Long polling**
   - Progress updates
   - Server-sent events

4. **Nowoczesne aplikacje**
   - FastAPI (native async)
   - Microservices
   - Cloud-native apps

5. **Wysokie concurrent users**
   - Wiele równoczesnych połączeń
   - ASGI = mniej resources

---

## 7. Czy są inne opcje poza WSGI i ASGI?

**NIE - to jedyne dwa standardy dla Python web applications.**

### Historia:

```
2003: PEP 333 → WSGI (synchroniczny standard)
  ↓
2016: async/await dodane do Pythona (PEP 492)
  ↓
2019: ASGI (asynchroniczny standard)
```

### Inne języki mają swoje standardy:

- **Node.js:** HTTP module (natywnie async, EventLoop)
- **Ruby:** Rack (odpowiednik WSGI)
- **Java:** Servlets, Spring
- **Go:** net/http (natywnie async, goroutines)
- **PHP:** PHP-FPM (FastCGI)

**W Pythonie: WSGI (sync) lub ASGI (async). Koniec listy.**

---

## 8. Kod ASGI aplikacji - przykład


In [ ]:
# Najprostsza ASGI aplikacja

async def application(scope, receive, send):
    """
    ASGI application - ASYNC funkcja!
    
    Parametry:
    - scope: dict z info o połączeniu (type: 'http'/'websocket', path, headers, etc.)
    - receive: async funkcja do odbierania danych od klienta
    - send: async funkcja do wysyłania danych do klienta
    """
    
    # Sprawdź typ połączenia
    if scope['type'] == 'http':
        # Obsługa HTTP request
        await send({
            'type': 'http.response.start',
            'status': 200,
            'headers': [[b'content-type', b'text/plain']],
        })
        
        await send({
            'type': 'http.response.body',
            'body': b'Hello from ASGI!',
        })
    
    elif scope['type'] == 'websocket':
        # Obsługa WebSocket
        await send({'type': 'websocket.accept'})
        
        while True:
            message = await receive()
            if message['type'] == 'websocket.receive':
                await send({
                    'type': 'websocket.send',
                    'text': f"Echo: {message['text']}"
                })
            elif message['type'] == 'websocket.disconnect':
                break

# Zauważ: wszystko async/await!
# Uruchom: uvicorn app:application

## 🎯 Kluczowe wnioski:

1. **WSGI** = synchroniczny, blokuje się na I/O, 1 worker = 1 request
2. **ASGI** = asynchroniczny, `await` na I/O (nie blokuje), 1 worker = wiele requestów
3. **CO jest async:** Tylko I/O operations (DB, HTTP, files), NIE CPU operations
4. **ASGI przewaga:** WebSockets, streaming, równoległe I/O, mniej workers
5. **Nie ma innych opcji** poza WSGI i ASGI dla Pythona

---

## 📚 Dalej:

**Następny notebook:** `03_uvicorn_gunicorn_serwery.ipynb`
- Czym dokładnie jest Uvicorn?
- Jak się ma do ASGI/WSGI?
- Django development server - co to?
- Alternatywy dla Uvicorn
- Gunicorn + Uvicorn workers

---